<a href="https://colab.research.google.com/github/Amani-K-code/SemProject_MLEngine/blob/main/Extracting_Kenyan_Foods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CELL 1 - IMPORTS

### FULL - KENYA-FOOD-COMPOSITION-TABLES-2018 Google drive link:
- https://drive.google.com/file/d/1dINVBm1g1zNYspB3E96dZrckucRPCNKW/view?usp=sharing

### EDITED/EXTRACTED PDF & CSV TABLE WITH THE TRADITIONAL KENYAN MEALS:

- https://drive.google.com/file/d/1BYcjmoSQiDtXRkYY_zD8Y3Z9vOzrJOgs/view?usp=sharing - pdf

- https://drive.google.com/file/d/1BYcjmoSQiDtXRkYY_zD8Y3Z9vOzrJOgs/view?usp=sharing - csv

In [5]:
import pandas as pd
import csv
import re
import requests
import io

# from google.colab import drive
# drive.mount('/content/drive')

# CELL 2 - EXTACTING THE 3 NUTRIENT TABLES FROM THE CSV

In [6]:
path = "https://raw.githubusercontent.com/Amani-K-code/SemProject_MLEngine/refs/heads/main/MAIN-KENYA-FOOD-COMPOSITION-TABLES-2018.csv"

# Since the Main Table extracted is divided into block of 3
# e.g. (page 1 - energy, water, protein...; page 2 - Ca, Fe, Mg...; page 3 - Vitamins, Niacin,...) then continues to repeat the same block strucutre
# We make one list per nutrient table (e.g. Macro - Page 1, minerals - Page 2, vitamin - Page 3) and track through as we read.

rows_macro, rows_mineral, rows_vitamin = [], [], []
current_section = None

response = requests.get(path)
response.raise_for_status()

f = io.StringIO(response.text)
reader = csv.reader(f)

for row in reader:
    row_text = " ".join(row)

    # now we detetct which section we are in baseed on the key words beggining at each grouping of pages i.e Energy - page 1, Ca - Page 2, Vit A - Page 3)
    if "Energy" in row_text:
      current_section = "macro"
    elif "Ca" in row_text and "Fe" in row_text:
      current_section = "mineral"
    elif "Vit A" in row_text:
      current_section = "vitamin"



    code = row[0].strip() if row else ""
    if re.fullmatch(r"\d{5}", code) and current_section: # only keep real food rows(no strays)
        cleaned = [v for v in row if v.strip() != ""]     # drop stray blank cells

        # cleaned lists of cleaned orws
        if current_section == "macro":
          rows_macro.append(cleaned)
        elif current_section == "mineral":
          rows_mineral.append(cleaned)
        else:
          rows_vitamin.append(cleaned)

print(len(rows_macro), len(rows_mineral), len(rows_vitamin))


140 140 140


# CELL 3 - FIX THE KNOWN BROKEN ROWS

The name for these were either split or lost

In [7]:
name_fixes = {
    "15042": ("Qita", "(Maize and wheat flour pancake)"),
    "15090": ("Kimanga cha Viazi Vitamu", "(Mashed Sweet Potato & Black Beans)"),
    "15123": ("Oat Porridge (Uji","wa Shayiri)"),

}

for r in rows_macro:
  if r[0] in name_fixes and len(r) == 11:
    r[1] = r[1] + " " + r[2]
    del r[2]

  if r[0] == "15035" and len(r) == 9:
    r.insert(1, "Saget, Terere & Managu (Spider plant, Amaranth & African Nightshade leaves)")


# CELL 4 - BUILD THE 3 DATAFRAMES AND CONVERT NUMBERS

The 3 major dataframes - (macro, mineral, vitamin)

In [8]:
print ([r for r in rows_vitamin if len(r) > 13])
print([r for r in rows_macro if len(r) > 11])
print([r for r in rows_mineral if len(r) > 10])
# this is a check for rows that have names that are very long and got split into two cells and hence two rows
# e.g

# (Uji',
# 'wa Shayiri)',

[['15123', 'Oat Porridge (Uji', 'wa Shayiri)', '11', '11', '11', '5', '0.05', '0.08', '0.1', '3', '3', '0.14', '0'], ['15129', 'Steamed Rice', '(Wali wa Mvuke)', '0', '0', '0', '0', '0.02', '0.04', '0.4', '2', '2', '0', '0']]
[]
[['15042', 'Qita', '(Maize and Wheat flour pancake)', '24', '2.8', '28', '131', '105', '158', '1.81', '0'], ['15090', 'Kimanga cha Viazi Vitamu', '(Mashed Sweet Potato & Black Beans)', '32', '1.6', '38', '78', '388', '276', '0.63', '2'], ['15123', 'Oat Porridge (Uji', 'wa Shayiri)', '39', '0.4', '18', '77', '69', '14', '0.45', '5']]


In [9]:
macro_cols = ["Code", "Food Name", "Energy_(kJ)", "Energy_(kcal)", "Water (g)", "Protein (g)", "Fat (g)", "Carb (g)", "Fibre (g)", "Ash (g)"]
mineral_cols = ["Code", "Food Name", "Calcium (mg)", "Iron (mg)", "Magnesium (mg)", "Phosphorus (mg)", "Potassium (mg)", "Sodium (mg)", "Zinc (mg)", "Selenium (mcg)"]
vitamin_cols = ["Code", "Food Name", "VitA_RAE", "VitA_RE", "Retinol", "BetaCaratone", "Thiamin", "Riboflavin", "Niacin", "DietaryFolatEq", "FoodFolate", "VitB12", "VitC"]


df_macro = pd.DataFrame(rows_macro, columns=macro_cols)

# Some mineral names were split due to the length
mineral_name_fixes = {"15042","15090", "15123"}

for r in rows_mineral:
  if r[0] in mineral_name_fixes and len(r) == 11:
    r[1] = r[1] + " " + r[2]                # rejoin the duplicates
    del r[2]                                # delete the remainder pieces

mineral_rows_padded = [r + [None] * (10 - len(r)) if len(r) < 10 else r for r in rows_mineral]
df_mineral = pd.DataFrame(mineral_rows_padded, columns=mineral_cols)

# Some vitamin rows can be short or have missing data - so we pad the short ones with "None" Before building for cleanliness in the main dataframe
vitamin_name_fixes = {"15123", "15129"} # codes where name got split into 2 cells due to the length

for r in rows_vitamin:
    if r[0] in vitamin_name_fixes and len(r) == 14:
        r[1] = r[1] + " " + r[2]    # rejoining duplicates e.g "Oat Porridge (Uji" + "wa Shayiri)"
        del r[2]                    # removing remanders of the duplicate rows where the neame was split to fit two sections

vitamin_rows_padded = [r + [None] * (13 - len(r)) for r in rows_vitamin]
df_vitamin = pd.DataFrame(vitamin_rows_padded, columns=vitamin_cols)


for df in (df_macro, df_mineral, df_vitamin):
  for col in df.columns:
    if col not in ("Code", "Food Name"):
      df[col] = pd.to_numeric(df[col], errors="coerce")
      # text -> real numbers instead
      # blanks -> NaN

print(df_macro.shape, df_mineral.shape, df_vitamin.shape)

(140, 10) (140, 10) (140, 13)


NOTE:

- One duplicate food code (15066) was found in the source document, assigned to two different foods." (i.e Biryani stew)
- So we decided to drop one of the duplicate 15066 entry from each table, to avoid the inconsitency caused at the final merge step

In [10]:
# Here we drop one of the two 15066 entries from each table , so that every code is unique and the main merge in cell 5 does not cause inconsistencies in the comuns + rows count

for df in (df_macro, df_mineral, df_vitamin):
    df.drop_duplicates(subset="Code", keep="first", inplace=True)

print(df_macro.shape, df_mineral.shape, df_vitamin.shape)


(139, 10) (139, 10) (139, 13)


# CELL 5 - Merging the three dataframes into the master/main df

i.e
(df_macro + df_mineral + df_vitamin) => **df_main**

In [12]:
# Merging the first two dataframes (macro + minerals), then merging the last dataframe ("i.e vitamins" to the master dataframe)
df_main = df_macro.merge(df_mineral.drop(columns="Food Name"), on="Code", how="left")
df_main = df_main.merge(df_vitamin.drop(columns="Food Name"), on="Code", how="left")


print(df_main)
df_main.head(10)

      Code                                  Food Name  Energy_(kJ)  \
0    15001            Uji wa mahindi (Maize Porridge)          221   
1    15002  Sorghum, Finger Millet and Maize Porridge          174   
2    15003                  Kaimati (Fried Dumplings)         1800   
3    15004                Mahamri (Swahili Doughnuts)         1730   
4    15005                    Whole Maize Flour Ugali          596   
..     ...                                        ...          ...   
134  15138                          Hydrabadi Biryani          819   
135  15139                           Chick Peas Curry          524   
136  15140               Bhature (Fried Indian Bread)         1410   
137  15141                Pumpkins with Peanut Butter          359   
138  15142                Firinda (Skinned bean Stew)          390   

     Energy_(kcal)  Water (g)  Protein (g)  Fat (g)  Carb (g)  Fibre (g)  \
0               52       87.5          1.5      1.1       8.5        1.1   
1      

,Code,Food Name,Energy_(kJ),Energy_(kcal),Water (g),Protein (g),Fat (g),Carb (g),Fibre (g),Ash (g),...,VitA_RE,Retinol,BetaCaratone,Thiamin,Riboflavin,Niacin,DietaryFolatEq,FoodFolate,VitB12,VitC
0,15001,Uji wa mahindi (Maize Porridge),221,52,87.5,1.5,1.1,8.5,1.1,0.3,...,7.0,7.0,3.0,0.02,0.05,0.2,7.0,7.0,0.09,0.0
1,15002,"Sorghum, Finger Millet and Maize Porridge",174,41,89.8,1.3,0.9,6.3,1.3,0.4,...,7.0,7.0,3.0,0.02,0.05,0.3,5.0,5.0,0.09,0.0
2,15003,Kaimati (Fried Dumplings),1800,429,18.8,4.6,21.8,52.8,1.6,0.4,...,30.0,30.0,1.0,0.33,0.25,2.0,NaN,76.0,0.25,0.0
3,15004,Mahamri (Swahili Doughnuts),1730,413,22.8,6.0,22.1,46.6,2.1,0.4,...,42.0,41.0,1.0,0.43,0.30,2.7,NaN,60.0,0.34,0.1
4,15005,Whole Maize Flour Ugali,596,141,64.5,3.3,1.8,26.0,3.8,0.5,...,0.0,0.0,0.0,0.08,0.04,0.8,20.0,20.0,0.00,0.0
5,15006,Maize and Finger Millet Flour Ugali,562,133,64.0,3.1,1.3,24.0,6.6,1.0,...,0.0,0.0,0.0,0.07,0.02,1.2,24.0,24.0,0.00,0.0
6,15007,"Maize, Red Sorghum and Finger Millet Ugali",543,129,66.2,3.1,1.4,23.3,5.3,0.8,...,0.0,0.0,0.0,0.07,0.03,1.0,20.0,20.0,0.00,0.0
7,15008,Cassava and Red Sorghum Ugali,489,116,69.6,2.1,0.8,23.4,3.4,0.7,...,0.0,0.0,3.0,0.05,0.04,0.7,14.0,14.0,0.00,0.5
8,15009,Refined Maize Flour Ugali,626,148,62.8,3.4,1.9,27.3,4.0,0.6,...,0.0,0.0,0.0,0.09,0.04,0.8,21.0,21.0,0.00,0.0
9,15010,"Sorghum, Maize flour & Finger Millet Ugali",634,150,61.0,1.9,0.7,32.1,3.6,0.7,...,0.0,0.0,1.0,0.04,0.02,0.7,11.0,11.0,0.00,0.0


# CELL 6 - Saving extract csv to Google drive

In [14]:
# df_main.to_csv("/content/drive/MyDrive/KENYA-FOOD-COMPOSITION-TABLES-2018/df_main_clean_extract.csv", index=False)
# index=False ... avoids having an extra column with indexes to the rows
# This was done and saved on my local Google drive initally.